## Unit 13. 參數估計 課堂作業
- 課程編號: CHEXXXX
- 學年度: 114下
- 上課時間: 每週四 09:00-12:00
-
- 指導教授: ＯＯＯ 教授
- 學生姓名: ＯＯＯ
- 學號: m12345678
- email帳號: fcu.m12345678@gmail.com

---
### 0. 環境設定

In [ ]:
from pathlib import Path
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'Unit13_Homework'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

try:
    shortcut_path = '/content/ChemE-3502'
    os.remove(shortcut_path)
except (FileNotFoundError, OSError):
    pass

if IN_COLAB:
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3502')
    os.symlink(source_path, shortcut_path)
    shortcut_path = Path(shortcut_path)
    if source_path.exists():
        NOTEBOOK_DIR = shortcut_path / 'Unit13'
        OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
        FIG_DIR      = OUTPUT_DIR / 'figs'
    else:
        print("⚠️ 找不到雲端 ChemE-3502 路徑，請確認資料夾名稱是否正確")
else:
    NOTEBOOK_DIR = Path.cwd()
    OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
    FIG_DIR      = OUTPUT_DIR / 'figs'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

---
### 1. 載入套件

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from scipy import linalg, optimize, integrate

# 繪圖樣式設定
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 10,
    'lines.linewidth': 2,
    'axes.unicode_minus': False,
})

print("✓ 套件載入完成")
print(f"  numpy  版本: {np.__version__}")
import scipy
print(f"  scipy  版本: {scipy.__version__}")
import matplotlib
print(f"  matplotlib 版本: {matplotlib.__version__}")

---
### **I. 練習題: 化工製程模式之參數估計**
本作業涵蓋三個化工製程參數估計問題，涉及**線性化最小平方法**、**非線性最小平方法**及**結合ODE求解之參數估計**三大核心技術。

#### 資料集說明
以下三個子題，各附有對應之實驗數據，資料均已內嵌於程式碼儲存格中，無須額外下載。

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# 問題一: Arrhenius 方程式 - 反應速率常數與溫度關係
# ============================================================
# 反應速率常數 k (1/min) 與 溫度 T (K)
k_data = np.array([0.0014, 0.0026, 0.0047, 0.0083, 0.014, 0.023, 0.038, 0.059, 0.090])
T_data = np.array([300, 310, 320, 330, 340, 350, 360, 370, 380])   # K

print("=" * 55)
print("問題一資料: Arrhenius 反應速率常數")
print("=" * 55)
df1 = pd.DataFrame({'T (K)': T_data, 'k (1/min)': k_data})
print(df1.to_string(index=False))

# ============================================================
# 問題二: 基質抑制模式 - 比生長速率
# ============================================================
# 比生長速率 mu (hr^-1) 與 基質濃度 x (g/L)
mu_data = np.array([0.24, 0.27, 0.34, 0.35, 0.35, 0.34, 0.33, 0.22])
x_data  = np.array([0.10, 0.15, 0.25, 0.50, 0.75, 1.00, 1.50, 3.00])  # g/L

print("\n" + "=" * 55)
print("問題二資料: 生化反應比生長速率")
print("=" * 55)
df2 = pd.DataFrame({'x (g/L)': x_data, 'mu (hr^-1)': mu_data})
print(df2.to_string(index=False))

# ============================================================
# 問題三: 批次反應器 - 一次反應動力學
# ============================================================
# 時間 t (min) 與 反應物濃度 CA (kg-mol/m^3)
t_batch = np.array([0, 1, 2, 3, 4, 5])          # min
CA_data = np.array([8.47, 5.00, 2.95, 1.82, 1.05, 0.71])  # kg-mol/m^3

print("\n" + "=" * 55)
print("問題三資料: 批次反應器濃度衰減")
print("=" * 55)
df3 = pd.DataFrame({'t (min)': t_batch, 'CA (kg-mol/m^3)': CA_data})
print(df3.to_string(index=False))

print("\n✓ 三個問題的資料載入完成")

---
### **1. 線性化模式之參數估計 — Arrhenius 方程式**

反應速率常數 $k$ 與溫度 $T$ 之關係由 Arrhenius 方程式表示：

$$
k = A \exp\!\left(\frac{-E}{RT}\right)
$$

其中 $A$ 為頻率因子 (1/min)，$E$ 為活化能 (J/g-mol)，$R = 8.314$ J/(g-mol·K)。

**提示**：對方程式取自然對數可得線性關係 $\ln k = \ln A - \dfrac{E}{RT}$，令 $Y = \ln k$，$X = \dfrac{1}{T}$，即可用線性最小平方法求解。

**要求**：
- 建構設計矩陣 $\mathbf{A}$ 並以 `scipy.linalg.lstsq()` 求解 $E$ 和 $A$
- 繪製 $\ln k$ vs $1/T$ 散點圖與擬合直線
- 繪製 $k$ vs $T$ 實驗值與模式預測值之比較圖
- 計算並輸出誤差平方和 $J$ (基於原始 $k$ 值)

---
### **2. 非線性參數估計與置信區間 — 基質抑制模式**

某生化反應之比生長速率 $\mu \, (\mathrm{hr}^{-1})$ 與基質濃度 $x \, (\mathrm{g/L})$ 之關係**無法**以簡單的 Monod 模式描述（請先說明原因），改以**基質抑制模式 (substrate inhibition model)** 表達：

$$
\mu = \frac{\mu_{\max} \, x}{K_m + x + K_1 x^2}
$$

其中 $\mu_{\max}$、$K_m$、$K_1$ 為未知參數。

**要求**：
- **2.1** 先以 Monod 模式（ $\mu = \mu_{\max} x / (K_m + x)$ ，上習題 7-3-2 之結果）嘗試擬合此資料，並說明為何不適用
- **2.2** 以 `scipy.optimize.curve_fit()` 估計基質抑制模式之三個參數
- **2.3** 由協方差矩陣 `pcov` 計算各參數之 $95\%$ 置信區間 ( $\pm 1.96\,\sigma$ )
- **2.4** 繪製實驗數據、Monod 模式預測、基質抑制模式預測之比較圖
- **2.5** 輸出參數估計值及置信區間，並說明哪個模式較適合

#### 2.1 先以 Monod 模式擬合（說明不適用原因）

**提示 (Monod 模式線性化)**：Monod 模式 $\mu = \mu_{\max} x / (K_m + x)$ 可整理為

$$
\mu_{\max} x - K_m \mu = \mu x
$$

令 $Y = \mu x$，設計矩陣各行為 $[x, \, -\mu]$，即可用線性最小平方法求 $\mu_{\max}$ 和 $K_m$。

#### 2.2–2.5  基質抑制模式之非線性參數估計與置信區間

**提示**：
- 以 `scipy.optimize.curve_fit(fun, x_data, mu_data, p0=[...])` 估計參數，返回 `popt, pcov`
- 各參數標準差 $\sigma_i = \sqrt{\mathrm{pcov}[i,i]}$，$95\%$ 置信區間為 $p_i \pm 1.96\,\sigma_i$
- 繪圖時請以連續 $x$ 值產生平滑曲線

---
### **3. 結合 ODE 求解之參數估計 — 批次反應器一次反應**

利用批次反應器進行 $\mathrm{A \rightarrow B}$ 之一次反應，其速率方程式為：

$$
\frac{dC_A}{dt} = -k C_A
$$

在起始濃度 $C_{A0} = 8.47 \, \mathrm{kg\text{-}mol/m^3}$ 下，由實驗數據估測反應速率常數 $k$。

**要求**：
- **3.1 線性化方法**：對 $C_A$ 之解析解 $C_A(t) = C_{A0} e^{-kt}$ 取自然對數，得 $\ln C_A = \ln C_{A0} - kt$；使用線性最小平方法估計 $k$ 值
- **3.2 非線性方法**：直接以 `scipy.optimize.curve_fit()` 擬合 $C_A(t) = C_{A0} e^{-kt}$ 之非線性模式，同時估計 $k$ 與 $C_{A0}$，並計算 $95\%$ 置信區間
- **3.3 ODE 驗證**：以估計所得之 $k$ 值，使用 `scipy.integrate.solve_ivp()` 求解 ODE 並繪圖驗證
- **3.4 繪圖比較**：在同一圖中比較實驗數據、線性化預測與 ODE 驗證之曲線
- **3.5 分析**：比較線性化方法與非線性方法所得之 $k$ 值差異，說明哪種方法較精確及原因

#### 3.1 線性化方法估計 $k$

**提示**：將解析解取對數得 $\ln C_A = \ln C_{A0} - k \cdot t$；設計矩陣 $\mathbf{A} = [1, \, -t]^T$，使用 `scipy.linalg.lstsq()` 求解 $[\ln C_{A0},\, k]$。

#### 3.2–3.5  非線性估計、ODE 驗證與分析

**提示 (ODE 求解)**：
```python
from scipy.integrate import solve_ivp
def ode_rxn(t, y, k):
    return [-k * y[0]]

sol = solve_ivp(ode_rxn, [t_batch[0], t_batch[-1]], [CA_data[0]],
                args=(k_est,), dense_output=True)
t_plot = np.linspace(0, 5, 200)
CA_plot = sol.sol(t_plot)[0]
```

---
### 💭 思考題

1. 線性化方法（取對數後套用最小平方法）與直接非線性最小平方法，兩者在統計意義上有何差異？哪個理論上更正確？
2. 協方差矩陣 `pcov` 非對角線元素代表什麼物理意義？
3. $95\%$ 置信區間愈寬代表什麼？可能的原因有哪些？
4. 基質抑制模式比 Monod 模式多一個參數，是否一定更好？如何判斷增加參數是否合理？
5. 若實驗數據點數很少（例如只有 3 個點），而模式參數有 4 個，會發生什麼問題？

---
## II. 額外加分作業（自由繳交）
- 學生可自由選擇是否繳交加分作業。（下週上課前完成 email 電子檔）
- 分數會加在最後總成績上，每份作業加 0.1 ~ 1.0 分。
- 分數加的不多，真的不一定要交加分作業，正常出席上課做好期末報告即可。
- 加分作業不一定要全部都做完，想繳交至少完成其中一項實驗即可。
- 請務必自行完成！你可以問 AI，問同學，問學長姊，甚至參考以前別人的作業，但一定要自行【吸收】【執行】【整理】結果！
- 不要貼 AI 的答案寄給老師看，你自己看就好。
- 如果系統自動比對發現作業內容（與他人雷同、原創性低、抄襲比例過高），作業的分數會 0 分。
- 如果你直接 100% 複製別人的作業繳交，你會直接被當掉（提供作業給別人複製的也一樣）。
- 老師看你作業要花時間，真的有心想寫加分作業，請好好自己做。

---
### 選擇你感興趣的化工資料集

以下加分實驗均基於課本第七章習題之化工實驗數據。嘗試自行完成其中一項或多項，並整理分析結果。建議選擇一個感興趣的主題，深入探索。

---
### **加分實驗 1: Antoine 方程式模式比較**

液體飽和蒸汽壓 $P \, (\mathrm{mmHg})$ 與溫度 $T \, (°C)$ 之關係以 Antoine 方程式表示如下：

$$
\ln P = A + \frac{B}{T + C}
$$

已知各實驗數據（習題 7-3-6 之苯蒸汽壓數據）如下：

| $T$ (°C) | $P$ (mmHg) | $T$ (°C) | $P$ (mmHg) | $T$ (°C) | $P$ (mmHg) |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 8.86 | 42.6 | 50.0 | 268.3 | 76.22 | 671.9 |
| 20.59 | 77.28 | 60.78 | 402.4 | 85.92 | 906.06 |
| 16.31 | 62.4 | 56.95 | 347.1 | 80.00 | 765.2 |
| 26.89 | 103.64 | 67.13 | 500.7 | 91.78 | 1074.6 |
| 26.1 | 98.4 | 63.33 | 437.0 | 81.30 | 789.2 |
| 35.19 | 149.4 | 74.03 | 627.9 | 97.69 | 1268.0 |
| 40.25 | 183.0 | 69.57 | 540.9 | 14.55 | 57.41 |
| 49.07 | 261.8 | 78.89 | 732.1 | 103.65 | 1489.1 |

**要求**：
- 以 `scipy.optimize.curve_fit()` 估計 $A$、$B$、$C$，起始值 $A \approx 16$、$B \approx -3000$、$C \approx 230$
- 計算並輸出各參數之 $95\%$ 置信區間
- 繪製實驗數據與模式擬合曲線之比較圖
- （進階）與 Clapeyron 模式 $\log P = A' + B'/T$ 比較，哪個模式更適合？

---
### **加分實驗 2: 醱酵程序動態模式之參數估計**

葡萄糖轉化為葡糖酸之醱酵程序，以四個狀態變數 ODE 模式描述：

$$\frac{dy_1}{dt} = b_1 y_1 \!\left(1-\frac{y_1}{b_2}\right), \quad \frac{dy_2}{dt} = \frac{b_3 y_1 y_4}{b_4+y_4} - 0.9082\,b_5\,y_2$$

$$\frac{dy_3}{dt} = b_5 y_2, \quad \frac{dy_4}{dt} = -1.011\frac{b_3 y_1 y_4}{b_4+y_4}$$

其中 $y_1$：細胞；$y_2$：內酯型葡萄糖酸；$y_3$：葡糖酸；$y_4$：葡萄糖（濃度單位詳見習題 7-3-7）。

起始條件：$y_1(0)=0.56, \quad y_2(0)=1.28, \quad y_3(0)=0.16, \quad y_4(0)=45.0$

實驗數據（部分）：

| $t$ (hr) | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| $y_1$ | 0.56 | 0.86 | 1.60 | 2.60 | 3.20 | 3.30 | 3.50 | 3.40 | 3.40 |
| $y_4$ | 45.0 | 43.0 | 38.0 | 33.0 | 25.0 | 16.5 | 9.00 | 4.00 | 2.00 |

**要求**：
- 以 `scipy.optimize.curve_fit()` 在外層進行參數搜尋，在模式函數內以 `solve_ivp()` 求解 ODE
- 估計 $b_1 \sim b_5$ 五個參數，計算 $95\%$ 置信區間
- 分別繪製 $y_1$ 和 $y_4$ 之實驗值與模式預測值比較圖

---
### **加分實驗 3: 多元線性迴歸 — Nusselt 數關係式**

某流體之 Nusselt 數 $Nu$ 與 Reynolds 數 $Re$ 之關係為：

$$
Nu = a \cdot \mathrm{Re}^b
$$

實驗數據如下：

| Re | 3520 | 6050 | 8400 | 9970 | 12520 | 14810 | 15900 | 18080 |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Nu | 11.6 | 18.1 | 23.5 | 26.9 | 32.2 | 36.8 | 39.0 | 43.2 |

**要求**：
- 對方程式取自然對數線性化：$\ln Nu = \ln a + b \cdot \ln Re$，建構設計矩陣並以 `scipy.linalg.lstsq()` 求解
- 以 `scipy.optimize.curve_fit()` 直接非線性求解做為比較
- 繪製 $Nu$ vs $Re$ 資料點與兩種方法之擬合曲線
- （進階）嘗試擴展至習題 7-3-11 之多元關係式 $Nu = \alpha \, Re^\beta \, Pr^\gamma \, (L/D)^a \, (\mu_b/\mu_s)^b$

---
# 想對老師說的話